In [1]:
!pip install --quiet qdrant-client "qdrant-client[fastembed]>=1.14.2" transformers torch pillow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 396.2/396.2 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.6/116.6 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.1/23.1 MB 68.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 324.1/324.1 kB 24.8 MB/s eta 0:00:00


In [2]:
import json
import os
import shutil
import numpy as np
import torch
from PIL import Image
from fastembed import TextEmbedding, SparseTextEmbedding
from qdrant_client import QdrantClient, models
from transformers import CLIPProcessor, CLIPModel

In [3]:
# ==========================================================
# CONFIGURATION
# ==========================================================
QDRANT_STORAGE = "./qdrant_store"

In [ ]:
#if we need to remove qdrant_storage
if os.path.exists(QDRANT_STORAGE):
       shutil.rmtree(QDRANT_STORAGE)
       print("🗑️ Removed existing Qdrant storage.")

In [4]:
#unzip qdrant_storage
!unzip qdrant_store_CLIP.zip

Archive:  qdrant_store_CLIP.zip
   creating: qdrant_store/
 extracting: qdrant_store/.lock      
  inflating: qdrant_store/meta.json  
   creating: qdrant_store/collection/
   creating: qdrant_store/collection/name_to_entry/
  inflating: qdrant_store/collection/name_to_entry/storage.sqlite  
   creating: qdrant_store/collection/hybrid_collection/
  inflating: qdrant_store/collection/hybrid_collection/storage.sqlite  
   creating: qdrant_store/collection/VDB_DP_IMAGES/
  inflating: qdrant_store/collection/VDB_DP_IMAGES/storage.sqlite  


In [5]:
!unzip BRRI.zip

Archive:  BRRI.zip
   creating: BRRI/Bacterial Leaf Blight/
  inflating: BRRI/Bacterial Leaf Blight/BLB_1946.jpeg  
  inflating: BRRI/Bacterial Leaf Blight/BLB_1953.jpeg  
  inflating: BRRI/Bacterial Leaf Blight/BLB_1954.jpeg  
  inflating: BRRI/Bacterial Leaf Blight/BLB_3184.jpeg  
  inflating: BRRI/Bacterial Leaf Blight/BLB_3188.jpeg  
  inflating: BRRI/Bacterial Leaf Blight/BLB_3189.jpeg  
  inflating: BRRI/Bacterial Leaf Blight/BLB_3190.jpeg  
  inflating: BRRI/Bacterial Leaf Blight/BLB_3192.jpeg  
  inflating: BRRI/Bacterial Leaf Blight/BLB_3193.jpeg  
  inflating: BRRI/Bacterial Leaf Blight/BLB_3197.jpeg  
  inflating: BRRI/Bacterial Leaf Blight/BLB_3199.jpeg  
  inflating: BRRI/Bacterial Leaf Blight/BLB_3201.jpeg  
  inflating: BRRI/Bacterial Leaf Blight/BLB_3202.jpeg  
  inflating: BRRI/Bacterial Leaf Blight/BLB_3206.jpeg  
  inflating: BRRI/Bacterial Leaf Blight/BLB_3207.jpeg  
  inflating: BRRI/Bacterial Leaf Blight/BLB_3208.jpeg  
  inflating: BRRI/Bacterial Leaf Blight/BLB_

In [6]:
clientQD = QdrantClient(path="./qdrant_store")

In [7]:
dense_model_name="sentence-transformers/all-MiniLM-L6-v2"
sparse_model_name="Qdrant/bm25"

dense_encoder = TextEmbedding(model_name=dense_model_name)
sparse_encoder = SparseTextEmbedding(model_name=sparse_model_name)

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 18 files:   0%|          | 0/18 [00:00<?, ?it/s]

In [8]:
# =========================================================
# LOAD CLIP
# =========================================================

clip_model = CLIPModel.from_pretrained(
    "openai/clip-vit-base-patch32"
)

processor = CLIPProcessor.from_pretrained(
    "openai/clip-vit-base-patch32"
)

config.json:   0%|          | 0.00/4.19k [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  605MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

model.safetensors: reconstructing file:   0%|          |  0.00B /  605MB            

model.safetensors: downloading bytes:           |  0.00B            

...

tokenizer_config.json:   0%|          | 0.00/592 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/862k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/525k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.22M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

In [9]:
from PIL import Image
import torch

def encode_image_clip(path):

    img = Image.open(path).convert("RGB")

    inputs = processor(
        images=img,
        return_tensors="pt"
    )

    with torch.no_grad():

        vision_outputs = clip_model.vision_model(
            pixel_values=inputs["pixel_values"]
        )

        pooled = vision_outputs.pooler_output

        feat = clip_model.visual_projection(
            pooled
        )

    feat = torch.nn.functional.normalize(
        feat,
        p=2,
        dim=-1,
    )

    feat = feat.squeeze().cpu().numpy()

    return feat.tolist()

In [10]:
from collections import defaultdict
import numpy as np
from qdrant_client import models


class MMRAGRetriever:

    def __init__(
        self,
        client,
        collection_name="hybrid_collection",
        img_collection_name="VDB_DP_IMAGES",
        name_collection="name_to_entry",
        dense_encoder=None,
        sparse_encoder=None,
        image_encoder_fn=None,
        dense_vector_name="dense",
        sparse_vector_name="sparse",
        rrf_k=10,
        fusion_depth = 30,
    ):

        self.client = client

        self.collection_name = collection_name
        self.img_collection_name = img_collection_name
        self.name_collection = name_collection

        self.dense_encoder = dense_encoder
        self.sparse_encoder = sparse_encoder
        self.image_encoder_fn = image_encoder_fn

        self.dense_vector_name = dense_vector_name
        self.sparse_vector_name = sparse_vector_name

        self.rrf_k = rrf_k
        self.fusion_depth = fusion_depth

    # ============================================================
    # Individual Retrievals
    # ============================================================

    def _dense_retrieve(self, query_text):

        dense_vec = list(self.dense_encoder.embed([query_text]))[0]

        response = self.client.query_points(
            collection_name=self.collection_name,
            query=dense_vec,
            using=self.dense_vector_name,
            limit=self.fusion_depth,
        )

        # print("Dense: ")
        # print(response.points)
        # print("\n")

        return response.points

    def _sparse_retrieve(self, query_text):

        sparse_vec = list(self.sparse_encoder.embed([query_text]))[0]

        response = self.client.query_points(
            collection_name=self.collection_name,
            query=models.SparseVector(**sparse_vec.as_object()),
            using=self.sparse_vector_name,
            limit=self.fusion_depth,
        )

        # print("BM25: ")
        # print(response.points)
        # print("\n")

        return response.points

    def _image_retrieve(self, query_image_path):

        image_vec = self.image_encoder_fn(query_image_path)
        image_vec = np.array(image_vec).flatten().tolist()

        response = self.client.query_points(
            collection_name=self.img_collection_name,
            query=image_vec,
            limit=self.fusion_depth,
        )

        # print("Image: ")
        # print(response.points)
        # print("\n")

        return response.points

    # ============================================================
    # External RRF
    # ============================================================

    def _rrf_fusion(self, ranked_lists):

        fused_scores = defaultdict(float)
        point_lookup = {}

        for ranked_list in ranked_lists:

            # ----------------------------------------------------
            # Keep BEST point per disease for this retrieval branch
            # ----------------------------------------------------
            best_per_name = {}

            for point in ranked_list:

                payload = point.payload or {}

                key = payload.get("name", "")

                if key == "":
                    continue

                if (
                    key not in best_per_name
                    or point.score > best_per_name[key].score
                ):
                    best_per_name[key] = point

            # ----------------------------------------------------
            # Apply RRF using disease-level ranking
            # ----------------------------------------------------
            deduped_points = sorted(
                best_per_name.values(),
                key=lambda x: x.score,
                reverse=True,
            )

            for rank, point in enumerate(deduped_points, start=1):

                key = point.payload.get("name", "")

                fused_scores[key] += 1.0 / (self.rrf_k + rank)

                # keep best overall representative point
                if (
                    key not in point_lookup
                    or point.score > point_lookup[key].score
                ):
                    point_lookup[key] = point

        # --------------------------------------------------------
        # Final fused ranking
        # --------------------------------------------------------
        ranked_items = sorted(
            fused_scores.items(),
            key=lambda x: x[1],
            reverse=True,
        )

        fused_points = []

        for key, fused_score in ranked_items[:self.fusion_depth]:

            point = point_lookup[key]

            point.score = float(fused_score)

            fused_points.append(point)

        return fused_points

    # ============================================================
    # Hybrid Retrieval Variants
    # ============================================================

    def _hybrid_retrieve(self, query_text):

        dense_points = self._dense_retrieve(query_text)

        sparse_points = self._sparse_retrieve(query_text)

        return self._rrf_fusion(
            [dense_points, sparse_points],
        )

    def _dense_image_retrieve(
        self,
        query_text,
        query_image_path,
    ):

        dense_points = self._dense_retrieve(
            query_text
        )

        image_points = self._image_retrieve(
            query_image_path
        )

        return self._rrf_fusion(
            [dense_points, image_points]
        )

    def _sparse_image_retrieve(
        self,
        query_text,
        query_image_path
    ):

        sparse_points = self._sparse_retrieve(
            query_text
        )

        image_points = self._image_retrieve(
            query_image_path
        )

        return self._rrf_fusion(
            [sparse_points, image_points]
        )

    def _full_mm_retrieve(
        self,
        query_text,
        query_image_path,
    ):

        dense_points = self._dense_retrieve(
            query_text
        )

        sparse_points = self._sparse_retrieve(
            query_text
        )

        image_points = self._image_retrieve(
            query_image_path
        )

        return self._rrf_fusion(
            [
                dense_points,
                sparse_points,
                image_points,
            ]
        )

    # ============================================================
    # Post Processing
    # ============================================================

    def _process_results(self, points, top_k):

        best_by_name = {}

        for point in points:

            payload = point.payload or {}

            name = payload.get("name", "unknown")
            typ = payload.get("type", "unknown")
            text = payload.get("text", "").strip()

            if (
                name not in best_by_name
                or point.score > best_by_name[name]["score"]
            ):

                best_by_name[name] = {
                    "id": point.id,
                    "score": point.score,
                    "name": name,
                    "type": typ,
                    "matched_sentence": text,
                }

        return sorted(
            best_by_name.values(),
            key=lambda x: x["score"],
            reverse=True,
        )[:top_k]

    # ============================================================
    # Metadata Attachment
    # ============================================================

    def _attach_metadata(self, results):

        for result in results:

            name = result["name"]

            if name == "unknown":

                result["sections_text"] = ""
                result["visual_symptoms"] = ""

                continue

            name_result = self.client.query_points(
                collection_name=self.name_collection,
                query_filter=models.Filter(
                    must=[
                        models.FieldCondition(
                            key="name",
                            match=models.MatchValue(value=name),
                        )
                    ]
                ),
                limit=1,
            )

            if name_result.points:

                payload = name_result.points[0].payload

                result["sections_text"] = payload.get(
                    "sections_text",
                    "",
                )

                result["visual_symptoms"] = payload.get(
                    "visual_symptoms",
                    "",
                )

            else:

                result["sections_text"] = ""
                result["visual_symptoms"] = ""

        return results

    # ============================================================
    # Public Retrieval API
    # ============================================================

    def retrieve(self, query_text = None, query_image_path = None, top_k=10):
        """
        Automatically selects:
            - hybrid retrieval (if both encoders)
            - dense retrieval (if only dense encoder)
            - sparse retrieval (if only sparse encoder)
        """

        if self.dense_encoder and self.sparse_encoder and self.image_encoder_fn:
            points = self._full_mm_retrieve(query_text, query_image_path)

        elif self.dense_encoder and self.sparse_encoder:
            points = self._hybrid_retrieve(query_text)

        elif self.dense_encoder and self.image_encoder_fn:
            points = self._dense_image_retrieve(query_text, query_image_path)

        elif self.sparse_encoder and self.image_encoder_fn:
            points = self._sparse_image_retrieve(query_text, query_image_path)

        elif self.dense_encoder:
            points = self._dense_retrieve(query_text)

        elif self.sparse_encoder:
            points = self._sparse_retrieve(query_text)

        elif self.image_encoder_fn:
            points = self._image_retrieve(query_image_path)

        else:
            raise ValueError("No encoder available! Provide dense or sparse encoder.")

        results = self._process_results(points, top_k)
        results = self._attach_metadata(results)

        return results

In [27]:
import pandas as pd

class RetrievalEvaluator:
    """
    OOP Evaluator for retrieval models.
    Works with any retriever having retriever.retrieve(query) → results.

    Computes:
    - RR (Reciprocal Rank)
    - Recall@K
    - Saves CSV
    """

    def __init__(self, retriever, csv_path, dataset_dir=""):
        self.retriever = retriever
        self.csv_path = csv_path
        self.dataset_dir = dataset_dir

    # ----------------------------------------------------------
    # Load and prepare data from CSV
    # CSV must contain cols:
    # file_name->image_file_name
    # name->class name (must match class folder name)
    # type->disease, pest, healthy
    # llm_desc->symptom description pre-generated from describer agent (to reduce exe time)
    # ----------------------------------------------------------
    def _load_data(self):
        df = pd.read_csv(self.csv_path)

        df = df[df["name"].notna()]       # remove NaN
        df["name"] = df["name"].astype(str).str.strip()
        df = df[df["name"] != ""]        # remove empty

        df["type"] = df["type"].fillna("").str.lower()
        df["llm_desc"] = df["llm_desc"].fillna("").astype(str).str.strip()
        df["file_name"] = df["file_name"].fillna("").astype(str).str.strip()

        return df.reset_index(drop=True)

    # ----------------------------------------------------------
    # Evaluate retrieval for each row
    # ----------------------------------------------------------
    def evaluate(self, top_k=10, query_type = "text", save_csv=True, csv_name="retrieval_eval.csv"):
        df = self._load_data()

        rr_list, rec_list = [], []

        for _, row in df.iterrows():
            query = row["llm_desc"]
            true_name = row["name"]
            query_image_path = None

            if query_type != "text":
              query_image_path = os.path.join(self.dataset_dir, row["name"], row["file_name"])

            if query_type == "text":
                results = self.retriever.retrieve(query, None, top_k=top_k)
            elif query_type == "image":
                results = self.retriever.retrieve(None, query_image_path, top_k=top_k)
            elif query_type == "mm":
                results = self.retriever.retrieve(query, query_image_path, top_k=top_k)

            names = [r["name"] for r in results]

            # Hit@K
            hit = int(true_name in names)

            # Reciprocal Rank
            if true_name in names:
                rank = names.index(true_name) + 1
                rr = 1 / rank
            else:
                rank = None
                rr = 0

            # Recall@K
            recall = 1 if hit else 0

            rr_list.append(rr)
            rec_list.append(recall)

        # --------------------------
        # Attach metrics to DF
        # --------------------------
        df["RR"] = rr_list
        df["Recall@K"] = rec_list

        # --------------------------
        # Aggregate Metrics
        # --------------------------
        results = self._aggregate_metrics(df)

        # Save
        if save_csv:
            df.to_csv(csv_name, index=False)

        return results, df

    # ----------------------------------------------------------
    # Type-wise (disease/pest/overall) aggregated metrics
    # ----------------------------------------------------------
    def _aggregate_metrics(self, df):
        metrics = ["RR", "Recall@K"]
        results = {}

        for m in metrics:
            results[f"Overall {m}"] = df[m].mean()

        return results


In [13]:
def print_metrics(title, results):

    print(f"\n==============================")
    print(title)
    print("==============================")

    for k, v in results.items():

        if v is None:
            print(f"{k}: None")
        else:
            print(f"{k}: {v:.4f}")


In [20]:
# =========================================================
# DENSE SEARCH EVALUATION
# =========================================================
print("Processing......")

retriever = MMRAGRetriever(
    client=clientQD,
    dense_encoder=dense_encoder,
)

evaluator_dense = RetrievalEvaluator(retriever, csv_path="/content/results_gemini-2.5-flash.csv")

results_dense, df_dense = evaluator_dense.evaluate(top_k=10)

print_metrics("Dense Retrieval", results_dense)

Processing......

Dense Retrieval
Overall RR: 0.4091
Overall Recall@K: 0.9051


In [28]:
# =========================================================
# SPARSE BM25 SEARCH EVALUATION
# =========================================================
print("Processing......")

retriever = MMRAGRetriever(
    client=clientQD,
    sparse_encoder=sparse_encoder,
)

evaluator_sparse = RetrievalEvaluator(retriever, csv_path="/content/results_gemini-2.5-flash.csv")

results_sparse, df_sparse = evaluator_sparse.evaluate(top_k=10)

print_metrics("Sparse BM25 Retrieval", results_sparse)

Processing......

Sparse BM25 Retrieval
Overall RR: 0.4396
Overall Recall@K: 0.8998


In [22]:
# =========================================================
# IMAGE SEARCH EVALUATION
# =========================================================
print("Processing......")

retriever = MMRAGRetriever(
    client=clientQD,
    image_encoder_fn=encode_image_clip,
)

evaluator_image = RetrievalEvaluator(retriever, csv_path="/content/results_gemini-2.5-flash.csv", dataset_dir="/content/BRRI")

results_image, df_image = evaluator_image.evaluate(top_k=10, query_type = "image")

print_metrics("Image Retrieval", results_image)

Processing......

Image Retrieval
Overall RR: 0.8674
Overall Recall@K: 1.0000


In [23]:
# =========================================================
# Dense+Sparse BM25 SEARCH EVALUATION
# =========================================================
print("Processing......")

retriever = MMRAGRetriever(
    client=clientQD,
    dense_encoder=dense_encoder,
    sparse_encoder=sparse_encoder
)

evaluator_hybrid = RetrievalEvaluator(retriever, csv_path="/content/results_gemini-2.5-flash.csv")

results_hybrid, df_hybrid = evaluator_hybrid.evaluate(top_k=10)

print_metrics("Dense+BM25 Retrieval", results_hybrid)


Processing......

Dense+BM25 Retrieval
Overall RR: 0.4676
Overall Recall@K: 0.9473


In [24]:
# =========================================================
# DENSE + IMAGE (MM)
# =========================================================
print("Processing......")

retriever = MMRAGRetriever(
    client=clientQD,
    dense_encoder=dense_encoder,
    image_encoder_fn=encode_image_clip,
)

evaluator_dense_image = RetrievalEvaluator(retriever, csv_path="/content/results_gemini-2.5-flash.csv", dataset_dir = "/content/BRRI")

results_dense_image, df_dense_image = evaluator_dense_image.evaluate(top_k=10, query_type = "mm")

print_metrics("Dense + Image Retrieval", results_dense_image)

Processing......

Dense + Image Retrieval
Overall RR: 0.7674
Overall Recall@K: 0.9965


In [25]:
# =========================================================
# SPARSE BM25 + IMAGE (MM)
# =========================================================
print("Processing......")

retriever = MMRAGRetriever(
    client=clientQD,
    sparse_encoder=sparse_encoder,
    image_encoder_fn=encode_image_clip,
)

evaluator_sparse_image = RetrievalEvaluator(retriever, csv_path="/content/results_gemini-2.5-flash.csv", dataset_dir = "/content/BRRI")


results_sparse_image, df_sparse_image = evaluator_sparse_image.evaluate(top_k=10, query_type = "mm")

print_metrics("Sparse BM25 + Image Retrieval", results_sparse_image)

Processing......

Sparse BM25 + Image Retrieval
Overall RR: 0.7725
Overall Recall@K: 1.0000


In [26]:
# =========================================================
# FULL MM-RAG (Dense + Sparse + Image)
# =========================================================
print("Processing......")

retriever = MMRAGRetriever(
    client=clientQD,
    dense_encoder=dense_encoder,
    sparse_encoder=sparse_encoder,
    image_encoder_fn=encode_image_clip,
)

evaluator_full_mm = RetrievalEvaluator(retriever, csv_path="/content/results_gemini-2.5-flash.csv", dataset_dir = "/content/BRRI")

results_full_mm, df_full_mm = evaluator_full_mm.evaluate(top_k=10, query_type = "mm")

print_metrics("Dense + BM25 + Image Retrieval", results_full_mm)

Processing......

Dense + BM25 + Image Retrieval
Overall RR: 0.7465
Overall Recall@K: 0.9930
